# CE survey data summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

# Load the data
df = pd.read_csv("/Users/eb2007/documents/phd/questionnaires_ML/CE_survey.csv")

# Display basic information
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "=" * 50)
print("COLUMN NAMES")
print("=" * 50)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

print("\n" + "=" * 50)
print("FIRST 5 ROWS")
print("=" * 50)
print(df.head())

print("\n" + "=" * 50)
print("DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n" + "=" * 50)
print("MISSING VALUES")
print("=" * 50)
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percent
})
print(missing_df[missing_df['Missing Count'] > 0])

# Check for numeric columns
numeric_cols = df.select_dtypes(include=['number']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

print("\n" + "=" * 50)
print("COLUMN TYPE SUMMARY")
print("=" * 50)
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")

if len(numeric_cols) > 0:
    print("\n" + "=" * 50)
    print("NUMERIC COLUMNS SUMMARY")
    print("=" * 50)
    print(df[numeric_cols].describe())
    
    # Visualize numeric distributions
    print("\n" + "=" * 50)
    print("NUMERIC COLUMNS DISTRIBUTIONS")
    print("=" * 50)
    if len(numeric_cols) <= 6:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        for i, col in enumerate(numeric_cols):
            if i < len(axes):
                df[col].hist(ax=axes[i], bins=20, alpha=0.7)
                axes[i].set_title(f'Distribution of {col}')
                axes[i].set_xlabel(col)
                axes[i].set_ylabel('Frequency')
        # Hide empty subplots
        for i in range(len(numeric_cols), len(axes)):
            axes[i].set_visible(False)
        plt.tight_layout()
        plt.show()
    else:
        # For many numeric columns, show correlation heatmap
        plt.figure(figsize=(10, 8))
        sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0)
        plt.title('Correlation Matrix of Numeric Columns')
        plt.tight_layout()
        plt.show()

print("\n" + "=" * 50)
print("CATEGORICAL COLUMNS ANALYSIS")
print("=" * 50)

for col in categorical_cols:
    print(f"\n--- {col} ---")
    value_counts = df[col].value_counts()
    print(f"Unique values: {len(value_counts)}")
    print(f"Most common value: {value_counts.index[0]} ({value_counts.iloc[0]} times)")
    print(f"Least common value: {value_counts.index[-1]} ({value_counts.iloc[-1]} times)")
    
    # Show top 10 values if there are many unique values
    if len(value_counts) > 10:
        print("Top 10 values:")
        print(value_counts.head(10))
    else:
        print("All values:")
        print(value_counts)
    
    # Check for missing values in this column
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        print(f"Missing values: {missing_count}")

# Sample data for inspection
print("\n" + "=" * 50)
print("SAMPLE DATA INSPECTION")
print("=" * 50)
print("Random 10 rows:")
print(df.sample(10))

# Check for any potential issues
print("\n" + "=" * 50)
print("DATA QUALITY CHECKS")
print("=" * 50)

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# Check for columns with all same values
constant_cols = []
for col in df.columns:
    if df[col].nunique() == 1:
        constant_cols.append(col)

if constant_cols:
    print(f"Columns with constant values: {constant_cols}")
else:
    print("No columns with constant values")

# Check for columns with very few unique values
low_cardinality_cols = []
for col in df.columns:
    if df[col].nunique() <= 5 and df[col].nunique() > 1:
        low_cardinality_cols.append(col)

if low_cardinality_cols:
    print(f"Columns with low cardinality (≤5 unique values): {low_cardinality_cols}")

print("\n" + "=" * 50)
print("EXPLORATION COMPLETE")
print("=" * 50)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load and clean the data
df = pd.read_csv("/Users/eb2007/documents/phd/questionnaires_ML/CE_survey.csv")
df_clean = df[~df['StartDate'].str.contains('Start Date|ImportId', na=False)].copy()

# Convert Q5 to numeric
df_clean['Q5'] = pd.to_numeric(df_clean['Q5'], errors='coerce')

print("=" * 60)
print("EDUCATION LEVEL - RAW DATA ANALYSIS")
print("=" * 60)

if 'Q5' in df_clean.columns:
    print("Raw education level codes:")
    education_counts = df_clean['Q5'].value_counts()
    for code, count in education_counts.items():
        print(f"  Code {code}: {count} responses")
    
    print(f"\nTotal responses: {len(df_clean['Q5'].dropna())}")
    
    # Check text responses for "Other" options
    if 'Q5_8_TEXT' in df_clean.columns:
        print("\nText responses for 'Other' education level:")
        other_responses = df_clean['Q5_8_TEXT'].dropna()
        for i, response in enumerate(other_responses):
            print(f"  {i+1}. {response}")

print("\n" + "=" * 60)
print("POSSIBLE CORRECTIONS")
print("=" * 60)

print("Based on the raw data, the most common response is Code 7 (16 people)")
print("This is likely NOT 'Doctorate/PhD' but probably:")
print("  - 'Bachelor's Degree' (most common in general population)")
print("  - 'Some College' or 'Associate Degree'")
print("  - Or another common education level")

print("\nTo fix this, we need to:")
print("1. Check your Qualtrics survey design")
print("2. See what the actual response options were")
print("3. Update the mapping accordingly")

print("\n" + "=" * 60)
print("RECOMMENDATION")
print("=" * 60)
print("Can you check your Qualtrics survey to see what the education")
print("question options were? The high number of 'Code 7' responses")
print("suggests it's likely a common education level, not PhD.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load and clean the data
df = pd.read_csv("/Users/eb2007/documents/phd/questionnaires_ML/CE_survey.csv")
df_clean = df[~df['StartDate'].str.contains('Start Date|ImportId', na=False)].copy()
df_clean['Q5'] = pd.to_numeric(df_clean['Q5'], errors='coerce')

print("=" * 60)
print("CORRECTED EDUCATION ANALYSIS")
print("=" * 60)

if 'Q5' in df_clean.columns:
    education_counts = df_clean['Q5'].value_counts()
    
    # CORRECT MAPPING based on the data and text responses
    correct_education_mapping = {
        1: 'Secondary school',
        2: 'High school graduate (high school diploma or equivalent)',
        3: 'Started university/college but did not finish',
        4: 'Apprenticeship or vocational qualification',
        5: 'Bachelor\'s degree or equivalent',
        6: 'Postgraduate degree or equivalent',
        7: 'Other',
        8: 'Prefer not to say'
    }
    
    print("Education Level Distribution:")
    total_education = len(df_clean['Q5'].dropna())
    for code, count in education_counts.items():
        if code == 9:
            label = "Unknown/Error code"
        else:
            label = correct_education_mapping.get(code, f'Code {code}')
        percentage = (count / total_education) * 100
        print(f"  {label}: {count} ({percentage:.1f}%)")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print("Based on the data:")
print("- Code 7 (16 people, 48.5%): 'Other' - includes PhD, EngD responses")
print("- Code 6 (7 people, 21.2%): 'Postgraduate degree or equivalent'")
print("- Code 8 (4 people, 12.1%): 'Prefer not to say'")
print("- Code 4 (2 people, 6.1%): 'Apprenticeship or vocational qualification'")
print("- Code 2 (1 person, 3.0%): 'High school graduate'")
print("- Code 5 (1 person, 3.0%): 'Bachelor\'s degree or equivalent'")
print("- Code 3 (1 person, 3.0%): 'Started university/college but did not finish'")
print("- Code 9 (1 person, 3.0%): Unknown/Error code")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print("The most common response is 'Other' (48.5%), which includes")
print("people with PhDs and EngDs who didn't select 'Postgraduate degree'.")
print("This suggests the 'Other' category captured specialized degrees.")